In [ ]:
# Thiết lập cho Google Colab
import os
if not os.path.exists('/content/ML-labs'):
    !git clone https://github.com/pie-12/ML-labs.git /content/ML-labs

os.chdir('/content/ML-labs/')
print("✅ Cấu hình dữ liệu thành công! Thư mục làm việc hiện tại:", os.getcwd())

# Lab 5 - Đánh giá Mô hình (Model Evaluation)

**Mục tiêu:**
1. Huấn luyện (Train) và đánh giá mô hình trên tập dữ liệu mất cân bằng (imbalanced dataset).
2. Sử dụng Matplotlib/Seaborn để minh họa: Heatmap Ma trận nhầm lẫn (Confusion Matrix), Đường cong ROC, và Đường cong Precision-Recall.
3. Hiểu tại sao Độ chính xác (Accuracy) lại có thể gây hiểu nhầm khi đánh giá dữ liệu mất cân bằng.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score

# Thiết lập style cho biểu đồ
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')

## 1. Tạo tập dữ liệu nhân tạo mất cân bằng

Chúng ta sẽ tạo ra một tập dữ liệu với 10.000 mẫu, trong đó 99% thuộc về lớp 0 (ví dụ: giao dịch bình thường) và 1% thuộc về lớp 1 (ví dụ: giao dịch gian lận).

In [ ]:
# Tạo tập dữ liệu mất cân bằng
X, y = make_classification(n_samples=10000, n_features=20, n_classes=2, 
                           weights=[0.99, 0.01], random_state=42)

print(f"Tổng số mẫu: {len(y)}")
print(f"Lớp 0 (Đa số): {sum(y == 0)}")
print(f"Lớp 1 (Thiểu số/Gian lận): {sum(y == 1)}")

## 2. Huấn luyện và Đánh giá Mô hình
Chia dữ liệu và huấn luyện một mô hình Hồi quy Logistic (Logistic Regression) đơn giản.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

### Tại sao Độ chính xác (Accuracy) lại gây hiểu nhầm với Dữ liệu mất cân bằng

Hãy kiểm tra độ chính xác của mô hình mà chúng ta vừa huấn luyện và so sánh với độ chính xác của một mô hình "ngốc" (dummy) luôn luôn dự đoán mọi thứ thuộc lớp đa số (Lớp 0).

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Độ chính xác của mô hình: {acc * 100:.2f}%")

dummy_preds = np.zeros_like(y_test)
dummy_acc = accuracy_score(y_test, dummy_preds)
print(f"Độ chính xác của mô hình ngốc (Luôn dự đoán 0): {dummy_acc * 100:.2f}%")

**Giải thích:**
Như chúng ta thấy, việc chỉ đơn giản dự đoán mọi giao dịch là bình thường (Lớp 0) mang lại độ chính xác ~99% vì dữ liệu quá mất cân bằng. Tuy nhiên, mô hình "ngốc" này hoàn toàn thất bại trong việc tìm ra các giao dịch gian lận thực sự (Lớp 1), mà đó lại là mục tiêu chính của bài toán. Do đó, **Độ chính xác (Accuracy) là một chỉ số không hiệu quả cho dữ liệu mất cân bằng**.

Thay vào đó, chúng ta nên xem xét Ma trận nhầm lẫn (Confusion Matrix), Precision, Recall và F1-score.

## 3. Bản đồ nhiệt Ma trận nhầm lẫn (Confusion Matrix Heatmap)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=['Dự đoán 0', 'Dự đoán 1'], 
            yticklabels=['Thực tế 0', 'Thực tế 1'])
plt.title('Ma trận nhầm lẫn (Confusion Matrix)', fontsize=16)
plt.show()

print("Báo cáo phân loại (Classification Report):\n", classification_report(y_test, y_pred))

## 4. Đường cong ROC (ROC Curve)

Đường cong ROC biểu diễn Tỉ lệ Dương tính thật (True Positive Rate - Recall) so với Tỉ lệ Dương tính giả (False Positive Rate - FPR) tại nhiều ngưỡng (thresholds) khác nhau.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Đường cong ROC (diện tích = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Phân loại ngẫu nhiên')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tỉ lệ Dương tính giả (FPR)', fontsize=12)
plt.ylabel('Tỉ lệ Dương tính thật (TPR / Recall)', fontsize=12)
plt.title('Đường cong Đặc trưng Hoạt động của Hệ thống (ROC)', fontsize=16)
plt.legend(loc="lower right")
plt.show()

## 5. Đường cong Precision-Recall

Đối với dữ liệu bị mất cân bằng nghiêm trọng, Đường cong Precision-Recall (PR) thường cung cấp nhiều thông tin hơn đường cong ROC. Nó làm nổi bật sự đánh đổi (trade-off) giữa Precision (Bao nhiêu phần trăm dự đoán dương tính là thực sự dương tính) và Recall (Bao nhiêu phần trăm các mẫu dương tính thực sự đã được tìm thấy).

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
avg_precision = average_precision_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, color='blue', lw=2, label=f'Đường cong PR (AP = {avg_precision:.2f})')
plt.xlabel('Recall (Độ thu hồi)', fontsize=12)
plt.ylabel('Precision (Độ chính xác chuẩn)', fontsize=12)
plt.title('Đường cong Precision-Recall', fontsize=16)
plt.legend(loc="lower left")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.show()